In [6]:
import pandas as pd
from sqlalchemy import text
import altair as alt
from altair import theme

from utils import get_db

In [4]:
engine = get_db("ipds")

q = """
WITH investor_tagged AS (
    SELECT
        das.sale_date,
        -- dense_rank() OVER (ORDER BY das.neighborhood) AS neighborhood,
        das.neighborhood,
        (spp.grantee_category = 0)::INT AS individual_purchase,
        -- (spc.grantor_prediction = 1)::INT AS grantor_prediction,
        avg_parcel_price
    FROM
        rocket.detroit_assessors_sales das
        -- JOIN sales_party_categories spc ON das.sale_id = spc.sale_id
        JOIN rocket.assessors_sales_avg app ON das.sale_id = app.sale_id
            AND das.parcel_id = app.parcel_id
        JOIN raw.detodp_assessors_20250522 pf ON das.parcel_id = pf.parcel_id
        JOIN rocket.sales_parties_predicted spp
            ON das.sale_id = spp.sale_id
    WHERE
        das.sale_date::DATE <= DATE '2025-12-31'
        AND avg_parcel_price > 100
        AND property_class_code = '401'
        AND is_improve > 0
        AND das.grantor <> 'DETROIT LAND BANK AUTHORITY'
        AND neighborhood IS NOT NULL
),
neighborhood_parcel_counts AS (
    SELECT neighborho AS neighborhood, COUNT(*) AS universe
    FROM raw.detodp_assessors_20250522 
    WHERE 
        property_c = '401'
        AND is_improve > 0
    GROUP BY neighborho
),
neighborhood_groups AS (
    SELECT
        it.neighborhood,
        EXTRACT(YEAR FROM sale_date::DATE) AS year,
        COUNT(*) AS sales,
        SUM(individual_purchase) AS individual_purchase, 
        MIN(avg_parcel_price) AS minimum_price,
        PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY avg_parcel_price) AS five_pct,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY avg_parcel_price) AS quarter,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY avg_parcel_price) AS median,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY avg_parcel_price) AS three_quarter,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY avg_parcel_price) AS ninetyfive_pct,
        MAX(avg_parcel_price) AS maximum_price
    FROM investor_tagged it
    GROUP BY EXTRACT(YEAR FROM sale_date::DATE), neighborhood
)
SELECT
    ng.neighborhood,
    year,
    sales,
    individual_purchase,
    sales - individual_purchase AS institutional_purchase,
    1000.0 * sales / universe AS sales_rate,
    1000.0 * individual_purchase / universe AS sales_rate_individ,
    1000.0 * (sales - individual_purchase) / universe AS sales_rate_institu,
    minimum_price,
    five_pct,
    quarter,
    median,
    three_quarter,
    ninetyfive_pct,
    maximum_price,
    universe
FROM neighborhood_groups ng
JOIN neighborhood_parcel_counts npc
    ON ng.neighborhood = npc.neighborhood;
"""

frame = pd.read_sql_query(text(q), engine.connect())

In [5]:
# Prep before the plotting
order = (frame.groupby("neighborhood")["three_quarter"]
         .last()
         .sort_values(ascending=False)
         .index)

mask = frame.groupby("neighborhood")["sales"].sum() > 10
long_enough_set = set(mask[mask].index)
included = [n for n in order if n in long_enough_set]
not_included = [n for n in order if n not in long_enough_set]

n_neighborhoods = len(included)

In [ ]:
# - [x] How many sales are happening?
# - [x] Where are investments happening
#    - [ ] what is happening in the 'study area'
# - [x] Who is purchasing?
# - [x] How many single-family homes in the area (Universe)

In [ ]:
@theme.register("d3_theme", enable=True)
def d3_theme() -> theme.ThemeConfig:
    sizes = 12, 14, 16, 18, 20, 24
    return {
        "font": "IBM Plex Sans",
        "autosize": {"contains": "content", "resize": True},
        "background": "#F3F2F1", # Why this?
        "config": {
            "font": "IBM Plex Sans",
            "axisX": {"labelFontSize": sizes[1], "titleFontSize": sizes[1]},
            "axisY": {"labelFontSize": sizes[1], "titleFontSize": sizes[1]},
            "headerColumn": {"labelFontSize": sizes[3]},
            "headerFacet": {"labelFontSize": sizes[1]},
            "headerRow": {"labelFontSize": sizes[1]},
            "legend": {"labelFontSize": sizes[0], "titleFontSize": sizes[1]},
            "text": {"fontSize": sizes[0]},
            "title": {"fontSize": sizes[-1]},
            "range": {
                "category": [
                    "#6596CF"
                    "#ECBA66", 
                    "#D3591C", 
                    "#58BFAC", 
                    "#CA7FCC", 
                    "#87AF3F", 
                ],
            }
        },
        "height": {"step": 28},
        "width": 320,
    }

In [8]:
FONT = "IBM Plex Sans"
AXIS_EXTENT = 30
CELL_W, TOP_H, BOT_H = 320, 150, 70     # default geometry for grid cells
N_COLS = 4
BUYER_COLS = ["sales_rate_individ", "sales_rate_institu"]


def _price_scale(df):
    return alt.Scale(domain=[0, df["ninetyfive_pct"].max()])

def _vol_scale(df):
    return alt.Scale(domain=[0, df[BUYER_COLS].max().max()])


def panel(single, *, price_scale=None, vol_scale=None,
          width=CELL_W, top_h=TOP_H, bot_h=BOT_H,
          show_legend=True, title=None):
    """Price band + grouped sales-rate bars for ONE neighborhood.

    single       DataFrame already filtered to one neighborhood (datetime `year`).
    price_scale,
    vol_scale    pass shared Scales to keep grid cells comparable; leave None
                 to auto-fit to `single` when used standalone.
    show_legend  draw the buyer-type legend (turn off for inner grid cells).
    title        panel heading; defaults to the neighborhood name.

    Returns a vconcat composite with NO top-level config, so it can be nested.
    """
    if price_scale is None:
        price_scale = _price_scale(single)
    if vol_scale is None:
        vol_scale = _vol_scale(single)
    if title is None:
        title = single["neighborhood"].iloc[0]
    n_parcels = int(single["universe"].max())

    x_hidden = alt.X("year:T", axis=None)

    band_outer = alt.Chart(single).mark_area(opacity=0.1, color="#6596CF").encode(
        x=x_hidden,
        y=alt.Y("five_pct:Q", title="Sale price", scale=price_scale,
                axis=alt.Axis(format="$,.0s", domain=False, ticks=False,
                              labelFont=FONT, titleFont=FONT,
                              minExtent=AXIS_EXTENT, maxExtent=AXIS_EXTENT,
                              labelFontSize=9, titleFontSize=10)),
        y2="ninetyfive_pct:Q",
    )
    band_inner = alt.Chart(single).mark_area(opacity=0.2, color="#6596CF").encode(
        x=x_hidden,
        y=alt.Y("quarter:Q", scale=price_scale), y2="three_quarter:Q",
    )
    line = alt.Chart(single).mark_line(color="#6596CF", strokeWidth=1.5).encode(
        x=x_hidden,
        y=alt.Y("median:Q", scale=price_scale),
    )
    label = alt.Chart(
        pd.DataFrame({"t": [f"{n_parcels:,} improved, single-family parcels"]})
    ).mark_text(align="left", baseline="top", font=FONT, fontSize=9,
                fontStyle="italic", color="#666").encode(
        x=alt.value(4), y=alt.value(4), text="t:N")

    top = (band_outer + band_inner + line + label).properties(
        width=width, height=top_h,
        title=alt.TitleParams(title, font=FONT, fontSize=12,
                              fontWeight="bold", anchor="middle"),
    )

    legend = alt.Legend(
        title=None, orient="top", labelFont=FONT, labelFontSize=9,
        labelExpr="datum.label == 'sales_rate_individ' "
                  "? 'Individual purchaser' : 'Business or Institutional purchaser'",
    ) if show_legend else None

    bottom = alt.Chart(single).transform_fold(
        BUYER_COLS, as_=["buyer", "rate"],
    ).mark_bar().encode(
        x=alt.X("year:T", title=None,
                axis=alt.Axis(labelAngle=0, tickCount=4, grid=False,
                              labelFont=FONT, labelFontSize=9)),
        xOffset="buyer:N",
        y=alt.Y("rate:Q", title="Sales rate", scale=vol_scale,
                axis=alt.Axis(domain=False, ticks=False,
                              labelFont=FONT, titleFont=FONT,
                              minExtent=AXIS_EXTENT, maxExtent=AXIS_EXTENT,
                              labelFontSize=9, titleFontSize=10)),
        color=alt.Color("buyer:N",
            scale=alt.Scale(domain=BUYER_COLS, range=["#ECBA66", "#C0772E"]),
            legend=legend),
    ).properties(width=width, height=bot_h)

    return alt.vconcat(top, bottom, spacing=8)


def small_multiples(data, neighborhoods, *, n_cols=N_COLS,
                    width=CELL_W, top_h=TOP_H, bot_h=BOT_H, title=""):
    """Tile per-neighborhood panels into a grid with shared scales."""
    price_scale, vol_scale = _price_scale(data), _vol_scale(data)
    cells = [
        panel(data[data["neighborhood"] == n],
              price_scale=price_scale, vol_scale=vol_scale,
              width=width, top_h=top_h, bot_h=bot_h,
              show_legend=(i == 0))            # legend on the first cell only
        for i, n in enumerate(neighborhoods)
    ]
    rows = [alt.hconcat(*cells[i:i + n_cols], spacing=30)
            for i in range(0, len(cells), n_cols)]
    return alt.vconcat(*rows, spacing=30).properties(
        title=alt.TitleParams(text=title, font=FONT, fontSize=20,
                              fontWeight="bold", anchor="middle", offset=20),
        padding={"top": 20, "left": 10, "bottom": 10, "right": 10},
    ).configure_view(stroke=None)


def show_single(single, **kwargs):
    """Render one panel on its own (adds the config a top-level chart needs)."""
    return panel(single, **kwargs).properties(
        padding={"top": 20, "left": 10, "bottom": 10, "right": 10},
    ).configure_view(stroke=None)

In [9]:
data = frame.copy()
data["year"] = pd.to_datetime(data["year"], format="%Y")
data = data.dropna(subset=["neighborhood"]).query("universe > 100")

neighborhoods = (
    data.sort_values("three_quarter", ascending=False)["neighborhood"].unique()
)

# small-multiple grid (shared scales, legend once)
chart = small_multiples(
    data, neighborhoods,
    title="Single-Family Home Price and Sales Rate Per 1,000 Parcels, "
          "by Neighborhood, Detroit 2011 - 2025",
)
chart

alt.VConcatChart(...)

In [10]:
ne = data[data["neighborhood"] == "Islandview"]

In [13]:
chart = alt.Chart(ne).mark_bar().transform_fold(
    ["individual_purchase", "institutional_purchase"],
).encode(
    x=alt.X("year:O", timeUnit="year"),
    #xOffset="key:N",
    y=alt.Y("value:Q"),
    color=alt.Color("key:N")
)

chart

alt.Chart(...)